In [0]:
!pip install openpyxl

In [0]:
import pandas as pd
import glob
import re
from pyspark.sql.functions import col, datediff, when, concat, coalesce, lit

In [0]:
def extract_date_from_filename(filename):
    """Extract date from filename in format 'dd.mm.yyyy'."""
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{4})\.xlsx', filename)
    if match:
        day, month, year = match.groups()
        return f'{year}-{month}-{day}'  # Format as YYYY-MM-DD
    return None

def add_reporting_date(file_paths):
    # Create an empty list to store DataFrames
    dfs = []

    for file in file_paths:
        # Extract the filename
        filename = file.split('/')[-1]
        
        # Read the Excel file
        df = pd.read_excel(file, dtype=str)
        
        # Extract date from filename for all files
        reporting_date = extract_date_from_filename(filename)
        df['Reporting Date'] = reporting_date
        
        dfs.append(df)
    return dfs

In [0]:
# main path
source_path = "/dbfs/mnt/stppeedp/ppeedp/landing/data0/staging/eag/ey/ap_automation/"

# Define the path to write the output 
destination_path = f'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/ageing'

companycode_path = source_path+'input_files/Company Code.xlsx'
ageing_filter_path = source_path+'input_files/Filters - Ageing.xlsx'
staff_mapping_path = source_path+'input_files/Staff Mapping.xlsx'
supplier_mapping_path = source_path+'input_files/Supplier Mapping.xlsx'

ageing_file_paths = source_path + 'ageing/Ageing*.xlsx'

In [0]:
# Reading Static Tables - CompanyCode, Filter Advances

# Reading CompanyCode and convert the Pandas DataFrame to a Spark DataFrame
companycode_df = pd.read_excel(companycode_path, dtype=str)
companycode = spark.createDataFrame(companycode_df)

# Reading Ageing filter and convert the Pandas DataFrame to a Spark DataFrame
ageing_filter_df = pd.read_excel(ageing_filter_path, dtype=str)
ageing_filter = spark.createDataFrame(ageing_filter_df)

# Reading Dynamic Tables - Staff Mapping, Supplier Mapping

# Reading Staff Mapping 
staff_mapping_df = pd.read_excel(source_path+'input_files/Staff Mapping.xlsx', dtype=str)
staff_mapping = spark.createDataFrame(staff_mapping_df)

# Reading Supplier Mapping 
supplier_mapping_df = pd.read_excel(source_path+'input_files/Supplier Mapping.xlsx', dtype=str)
supplier_mapping = spark.createDataFrame(supplier_mapping_df)

In [0]:
# Define the path to your Excel files
file_paths = glob.glob(ageing_file_paths)

# Add Reporting Date Column
dfs = add_reporting_date(file_paths)

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Convert the Pandas DataFrame to a Spark DataFrame
ageing_df = spark.createDataFrame(combined_df)

# Add the new columns
ageing_df = ageing_df.withColumn(
    "Difference", datediff(col("Reporting Date"), col("Net due date"))
).withColumn(
    "Ageing",
    when(col("Difference").isNull(), 'Unknown') \
    .when(col("Difference") <= 30, '0-30 Days') \
    .when((col("Difference") > 30) & (col("Difference") <= 60), '31-60 Days') \
    .when((col("Difference") > 60) & (col("Difference") <= 90), '61-90 Days') \
    .when((col("Difference") > 90) & (col("Difference") <= 365), '91-365 Days') \
    .otherwise('More than 1 year')
)

# Join with companycode to get Company_Name
ageing_df = ageing_df.join(companycode, ageing_df["Company Code"] == companycode["Company Code"], "left").drop(companycode["Company Code"])

# Join with document type to get Ageing Filter
ageing_df = ageing_df.join(ageing_filter, ageing_df["Document type"] == ageing_filter["Document Type"], "left").drop(ageing_filter["Document Type"]).drop("Document Description").withColumnRenamed("Document Flag","Document Filter") \
    .withColumn("Document Filter", coalesce(col("Document Filter"), lit(0)))

ageing_df = ageing_df.withColumn(
    "Conc CC Vendor", concat(col("Company Code"), col("Vendor"))
)

# Drop duplicates to keep only one row per CC_Code_Vendor
supplier_mapping_unique = supplier_mapping.dropDuplicates(["CC Code+Vendor"])

# Join with supplier_mapping_unique to get Vendor Mapping
ageing_df = ageing_df.join(supplier_mapping_unique, ageing_df["Conc CC Vendor"] == supplier_mapping_unique["CC Code+Vendor"], "left") \
    .drop(supplier_mapping_unique.Vendor).drop(supplier_mapping_unique["Company Code"]) \
    .drop("CC Code+Vendor", "Team Lead","Category","Vendor Status") \
    .withColumnRenamed("Manager", "Vendor Mapping") \
    .withColumn("Vendor Mapping", coalesce(col("Vendor Mapping"), lit('Unassigned'))) \
    .withColumn("Vendor type", coalesce(col("Vendor type"), lit('Other')))

# Drop duplicates to keep only one row per Employee Number
staff_mapping_unique = staff_mapping.dropDuplicates(["Employee Number"])

# Join with staff_mapping to get staff
ageing_df = ageing_df.join(staff_mapping_unique, ageing_df["User Name"] == staff_mapping_unique["Employee Number"], "left") \
    .drop("Employee Number", "Team Lead", "Line Manager","Status","Scope") \
    .withColumnRenamed("Employee Full Name", "Staff Mapping") \
    .withColumn("Staff Mapping", coalesce(col("Staff Mapping"), lit('Unassigned'))) \
    .withColumn("Ageing Type", when(col("Company Code Currency Value") < 0, "Invoice").otherwise("Credit Note"))

# Write the data to the destination
ageing_df.write.mode("overwrite").parquet(destination_path)
print(f"Data has been successfully processed and written at {destination_path}")